In [1]:
vdb_strategy_name="GROBID_HF_Paragraph"
db_strategy_name="grobid"

In [2]:
from episcope.db.in_memory_academic_db import InMemoryAcademicDB
db = InMemoryAcademicDB("full_WP2_db.json")

INFO:episcope.db.in_memory_academic_db:Loading backup from full_WP2_db.json


In [3]:
from episcope.vectordb.qdrant import QdrantDB
vdb = QdrantDB(collection="wp2_HF", url="http://localhost:6334")

INFO:httpx:HTTP Request: GET http://localhost:6334 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET http://localhost:6334/collections/wp2_HF "HTTP/1.1 200 OK"


In [5]:
import pandas as pd
df = pd.read_csv("CLASSIFICATION_qwen2.5vl.csv")
# df_filtered = 
data_analysis_papers = df[df['classification'] == 'DATA_ANALYSIS'].paper_id.to_list()

In [6]:
from episcope.rag.retrieval.semantic import SemanticRetriever
retriever = SemanticRetriever(vdb)

INFO:datasets:PyTorch version 2.6.0 available.
INFO:datasets:Polars version 1.34.0 available.
INFO:httpx:HTTP Request: POST http://localhost:6334/collections/wp2_HF/points/scroll "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:6334/collections/wp2_HF/points/scroll "HTTP/1.1 200 OK"
INFO:episcope.rag.retrieval.semantic:VectorDB is configured with embedding model: 'sentence-transformers/all-MiniLM-L6-v2'. Instantiating corresponding embedder for retrieval.
INFO:episcope.rag.embeddings.factory:Detected HuggingFace model 'sentence-transformers/all-MiniLM-L6-v2'. Creating HuggingFaceEmbedder.
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
INFO:sentence_transformers.SentenceTransformer:2 prompts are loaded, with the keys: ['query', 'text']


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
from episcope.rag.generation.llm_generator import LLMGenerator
from episcope.clients import GeminiClient
generator = LLMGenerator(model="phi3:3.8b") # "qwen2.5vl:3b"  phi3:3.8b 
# client = GeminiClient()
# generator = LLMGenerator(client=client, model="gemini-2.5-flash-lite")


In [16]:
from episcope.workflows import PrecisionMiner
from episcope.workflows.precision_miner import  FindDataSourcesConfig
# from episcope.config.miner import FindDataSourcesConfig
miner = PrecisionMiner(
    retriever, 
    generator, 
    strategy_name=db_strategy_name,
    config=FindDataSourcesConfig(),
    academic_db=db
)

In [17]:
list_results = []
for paper_id in data_analysis_papers[:1]:
    # if paper_id in ["2014_Biggerstaff_serial_interval"]:  #,"10.1101/2023.10.03.23296552"
    #     print(f"Skipping paper_id: {paper_id}")
    #     continue
    print(f"Processing paper_id: {paper_id}")
    meta = db.get_paper_metadata(paper_id, strategy_name=db_strategy_name)
    er = miner.run(paper_id=paper_id)

    list_inner_results = []
    result = {

        "paper_id": paper_id,
        "paper_title":meta.title,
        "doi":meta.doi,
        "description": er.description,
    }

    for i in er.items:
        inner_res = {
            **result,
            "name": i.name,
            "url": i.url,
            "explanation": i.explanation,
            "raw_text": i.raw_text,
        }

        list_inner_results.append(inner_res)
    list_results.extend(list_inner_results)

Processing paper_id: 2003_Lloyd-Smith_et_al_infectious_period


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: POST http://localhost:6334/collections/wp2_HF/points/search "HTTP/1.1 200 OK"


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: POST http://localhost:6334/collections/wp2_HF/points/search "HTTP/1.1 200 OK"


...

INFO:httpx:HTTP Request: POST http://localhost:6334/collections/wp2_HF/points/search "HTTP/1.1 200 OK"


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: POST http://localhost:6334/collections/wp2_HF/points/search "HTTP/1.1 200 OK"


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: POST http://localhost:6334/collections/wp2_HF/points/search "HTTP/1.1 200 OK"
E0000 00:00:1761824207.241435 29844961 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.


In [18]:
import pandas as pd
results_df = pd.DataFrame(list_results)
results_df

,paper_id,paper_title,doi,description,name,url,explanation,raw_text
0,2003_Lloyd-Smith_et_al_infectious_period,Curtailing transmission of severe acute respir...,10.1098/rspb.2003.2481,The following data sources were identified as ...,Riley et al. (2003),None,This reference is repeatedly cited for providi...,These results agree qualitatively with Riley e...
1,2003_Lloyd-Smith_et_al_infectious_period,Curtailing transmission of severe acute respir...,10.1098/rspb.2003.2481,The following data sources were identified as ...,Lipsitch et al. (2003),None,This reference is cited alongside Riley et al....,Recent analyses of SARS incidence data from Ho...
2,2003_Lloyd-Smith_et_al_infectious_period,Curtailing transmission of severe acute respir...,10.1098/rspb.2003.2481,The following data sources were identified as ...,Hong Kong Department of Health,None,This is cited as a source for the proportion o...,"As of early June 2003, HCWs have comprised ca...."
3,2003_Lloyd-Smith_et_al_infectious_period,Curtailing transmission of severe acute respir...,10.1098/rspb.2003.2481,The following data sources were identified as ...,World Health Organization,None,Cited for providing data on the proportion of ...,"As of early June 2003, HCWs have comprised ca...."
4,2003_Lloyd-Smith_et_al_infectious_period,Curtailing transmission of severe acute respir...,10.1098/rspb.2003.2481,The following data sources were identified as ...,Dwosh et al. (2003),None,This reference is used to support the discussi...,Infection of other patients has played a signi...
5,2003_Lloyd-Smith_et_al_infectious_period,Curtailing transmission of severe acute respir...,10.1098/rspb.2003.2481,The following data sources were identified as ...,Galvani et al. (2003),None,This reference is cited for reporting widely v...,"Thus, transmission rates are both disease-and ..."
6,2003_Lloyd-Smith_et_al_infectious_period,Curtailing transmission of severe acute respir...,10.1098/rspb.2003.2481,The following data sources were identified as ...,Mandavilli (2003),None,This reference is used to support the assumpti...,We assume that transmission in quarantine is r...
7,2003_Lloyd-Smith_et_al_infectious_period,Curtailing transmission of severe acute respir...,10.1098/rspb.2003.2481,The following data sources were identified as ...,Booth et al. (2003),None,Cited for providing data on the proportion of ...,"As of early June 2003, HCWs have comprised ca...."
8,2003_Lloyd-Smith_et_al_infectious_period,Curtailing transmission of severe acute respir...,10.1098/rspb.2003.2481,The following data sources were identified as ...,Leo et al. (2003),None,Cited for providing data on the proportion of ...,"As of early June 2003, HCWs have comprised ca...."
9,2003_Lloyd-Smith_et_al_infectious_period,Curtailing transmission of severe acute respir...,10.1098/rspb.2003.2481,The following data sources were identified as ...,Twu et al. (2003),None,Cited for providing data on the proportion of ...,"As of early June 2003, HCWs have comprised ca...."


In [ ]:
# results_df.to_csv("data_analysis_papers_data_sources_gemini-2.5-flash-lite_NO_SNIPPETS.csv", index=False)